## Incremental Data Ingestion with Auto Loader
- *Concept:* **Auto Loader** is a Databricks feature built on top of **Spark Structured Streaming** that incrementally and efficiently processes new data files as they arrive in cloud storage (ADLS Gen2, S3, GCS) without needing to manually track which files have already been loaded.
- *Why Auto Loader over Batch Reads?*
  - **Scalability:** Efficiently discovers millions of files using directory listing or file notification services.
  - **Exactly-Once Processing:** Automatically tracks processed files using **Checkpointing**, ensuring no duplicate data is ingested.
  - **Schema Inference & Evolution:** Automatically detects schemas and handles schema changes (new columns, data type mismatches) without breaking the pipeline.
  - **Format Identifier:** Auto Loader uses the `cloudFiles` source format in `spark.readStream`.

## Storage Setup (ADLS Gen2 Directories)
- Inside your existing ADLS Gen2 container (`mycontainer`), create a folder structure for your Auto Loader pipeline:
  - *Source Directory:* `abfss://mycontainer@storage_account_name.dfs.core.windows.net/autoloader/source/` (where incoming raw files like CSV/JSON/Parquet land).
  - *Destination Directory:* `abfss://mycontainer@storage_account_name.dfs.core.windows.net/autoloader/destination/` (for storing the target Delta table data, if external).
  - *Checkpoint & Schema Directory:* `abfss://mycontainer@storage_account_name.dfs.core.windows.net/autoloader/checkpoint/` (stores streaming state, processed file logs, and inferred schema metadata).

## Reading Data Incrementally with Auto Loader (PySpark)
- *Concept:* Use `spark.readStream.format("cloudFiles")` to read files incrementally from your ADLS Gen2 source folder.
- *Key Options:*
  - `cloudFiles.format`: Specifies the incoming raw file format (e.g., `csv`, `json`, `parquet`).
  - `cloudFiles.schemaLocation`: Path where Auto Loader stores the inferred schema so it doesn't have to re-infer it on every run.
  - `cloudFiles.schemaEvolutionMode`: Controls how new columns are handled (default is `addNewColumns`).
- *PySpark Example:*
```python
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "abfss://mycontainer@storage_account_name.dfs.core.windows.net/autoloader/checkpoint")
    .option("header", "true")
    .load("abfss://mycontainer@storage_account_name.dfs.core.windows.net/autoloader/source")
)
```

## Writing Incrementally to a Delta Table (`writeStream`)
- *Concept:* Write the streaming DataFrame into a Unity Catalog Delta table (or external Delta path) using `writeStream`.
- *Key Options:*
  - `checkpointLocation`: **Mandatory** for fault tolerance. Tracks which files have already been processed so re-running the job only picks up newly arrived files.
  - `mergeSchema`: Set to `"true"` to allow new columns detected by Auto Loader to be merged into the target Delta table.
  - `trigger(availableNow=True)`: Processes all currently available new files in micro-batches and then **automatically stops** the stream (ideal for scheduled batch-like incremental jobs to save compute costs).
- *PySpark Example (Writing to a Unity Catalog Table):*
```python
(
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://mycontainer@storage_account_name.dfs.core.windows.net/autoloader/checkpoint")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("man_cata.man_schema.autoloader_table")
)
```
- *Alternative (Writing to an External ADLS Path):*
```python
(
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://mycontainer@storage_account_name.dfs.core.windows.net/autoloader/checkpoint")
    .trigger(availableNow=True)
    .start("abfss://mycontainer@storage_account_name.dfs.core.windows.net/autoloader/destination")
)
```

## How Incremental Ingestion Works (Testing Step-by-Step)
1. **Initial Load:** Upload `file1.csv` into the `autoloader/source` folder in ADLS Gen2 and run the `readStream` / `writeStream` cells.
   - Auto Loader infers the schema, saves it in `schemaLocation`, loads `file1.csv` into the Delta table, and records `file1.csv` as processed in `checkpointLocation`.
2. **Verify Data:** Query the Delta table to confirm the initial rows are loaded:
```sql
SELECT * FROM man_cata.man_schema.autoloader_table;
```
3. **Incremental Load:** Upload a second file (`file2.csv`) into the same `autoloader/source` folder and re-run the stream.
   - Auto Loader checks the `checkpointLocation`, **skips `file1.csv`**, and ingests **only `file2.csv`**.
   - Querying the table again shows records from both files with zero duplicates.

## Schema Evolution & `_rescued_data` Column
- *Rescued Data Column (`_rescued_data`):* Auto Loader automatically adds a `_rescued_data` JSON column to capture any data that doesn't match the expected schema (e.g., data type mismatches, extra unexpected columns when evolution is disabled, or case mismatches) so no data is ever silently dropped.
- *Schema Evolution Modes (`cloudFiles.schemaEvolutionMode`):*
  - `addNewColumns` *(Default)*: Fails the current stream when a new column is detected, updates the schema in `schemaLocation`, and automatically ingests the new column on the next run.
  - `rescue`: Never fails the stream; places all unexpected columns into the `_rescued_data` column.
  - `failOnNewColumns`: Strictly fails the stream if a new column appears (requires manual intervention).
  - `none`: Ignores new columns completely without failing or rescuing.